# Synthetic Swath Evaluation Report

This notebook audits and visualizes one completed `synthetic_eval.cli` run. It is designed for the swath-conditioned concat evaluation with saved tensors.

It checks:

- whether the run finished;
- whether expected result files exist;
- whether saved `.npz` tensors have the expected shapes and finite values;
- whether the observed tensor is consistent with `truth * mask`;
- aggregate RMSE/CRPS/coverage/spread-skill;
- rank histograms and calibration diagnostics;
- example ensemble samples and saved plot files.

The default output directory is the 30-day swath run with four track masks per day.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

try:
    import pandas as pd
except ImportError:
    pd = None

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/mnt/sciml/a.sadreev/sea_ice_data"))
OUT_DIR = Path(
    os.environ.get(
        "SYNTH_EVAL_OUT",
        DATA_ROOT / "synthetic_eval_swath_30days_x32_r4_saved",
    )
)

if not OUT_DIR.exists():
    candidates = sorted(DATA_ROOT.glob("synthetic_eval*"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"No synthetic_eval output directories found under {DATA_ROOT}")
    print(f"Configured OUT_DIR does not exist: {OUT_DIR}")
    OUT_DIR = candidates[0]
    print(f"Using latest synthetic_eval directory instead: {OUT_DIR}")

PLOTS_DIR = OUT_DIR / "plots"
ARRAYS_DIR = OUT_DIR / "arrays"
SAMPLES_DIR = OUT_DIR / "samples"

print("OUT_DIR    =", OUT_DIR)
print("PLOTS_DIR  =", PLOTS_DIR)
print("ARRAYS_DIR =", ARRAYS_DIR)
print("SAMPLES_DIR=", SAMPLES_DIR)

In [ ]:
def read_json(path: Path):
    with open(path) as f:
        return json.load(f)


def read_csv_table(path: Path):
    if not path.exists():
        return None
    if pd is not None:
        return pd.read_csv(path)
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def to_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default


def table_rows(table):
    if table is None:
        return []
    if pd is not None and hasattr(table, "to_dict"):
        return table.to_dict("records")
    return table


def display_table(table, n=20):
    if table is None:
        print("missing table")
        return
    if pd is not None and hasattr(table, "head"):
        display(table.head(n))
    else:
        display(table[:n])


def show_png(path: Path, width=980):
    if path.exists():
        display(Markdown(f"**{path.relative_to(OUT_DIR)}**"))
        display(Image(filename=str(path), width=width))
    else:
        print("missing:", path)


def show_many(pattern: str, width=980, limit=None):
    paths = sorted(PLOTS_DIR.glob(pattern))
    if limit is not None:
        paths = paths[:limit]
    if not paths:
        print(f"no plots matched {pattern}")
    for path in paths:
        show_png(path, width=width)


def log_tail(path: Path, n=80):
    if not path.exists():
        return ""
    return "\n".join(path.read_text(errors="replace").splitlines()[-n:])


def looks_finished(log_text: str):
    return '"output_dir"' in log_text and '"n_cases"' in log_text and '"n_rows"' in log_text


def expected_condition_count(metadata):
    mask_types = metadata.get("mask_types", [])
    densities = metadata.get("densities", [])
    noise_levels = metadata.get("noise_levels", [])
    swath_repeats = int(metadata.get("swath_repeats", 1) or 1)
    count = 0
    for mask_type in mask_types:
        repeats = swath_repeats if mask_type == "swath" else 1
        count += repeats * max(1, len(densities)) * max(1, len(noise_levels))
    return count


print("top-level output files:")
for path in sorted(OUT_DIR.glob("*")):
    kind = "dir" if path.is_dir() else "file"
    print(f"{kind:4s} {path.name}")

## 1. Run Integrity

In [ ]:
metadata_path = OUT_DIR / "metadata.json"
metadata = read_json(metadata_path) if metadata_path.exists() else {}
log_text = log_tail(OUT_DIR / "run.log", n=120)

required_files = [
    OUT_DIR / "metadata.json",
    OUT_DIR / "aggregate_metrics.csv",
    OUT_DIR / "per_case_metrics.csv",
    OUT_DIR / "spread_skill_bins.csv",
    ARRAYS_DIR / "rank_histograms.npz",
]

tensor_paths = sorted(SAMPLES_DIR.glob("**/*.npz"))
n_cases = int(metadata.get("n_cases") or len(metadata.get("selected_case_indices", [])) or 0)
save_tensor_limit = metadata.get("save_tensor_limit")
n_saved_cases_expected = n_cases if save_tensor_limit is None else min(n_cases, int(save_tensor_limit))
n_conditions = expected_condition_count(metadata) if metadata else 0
expected_tensor_files = n_saved_cases_expected * n_conditions if metadata.get("save_tensors") else 0

checks = []
checks.append(("metadata exists", metadata_path.exists()))
checks.append(("run appears finished", looks_finished(log_text)))
for path in required_files:
    checks.append((f"exists: {path.name if path.parent == OUT_DIR else path.parent.name + '/' + path.name}", path.exists()))
if metadata.get("save_tensors"):
    checks.append(("samples directory exists", SAMPLES_DIR.exists()))
    checks.append((f"tensor files count == expected ({expected_tensor_files})", len(tensor_paths) == expected_tensor_files))
    checks.append(("all expected saved cases have at least one archive", len({p.name.split('_')[0] for p in tensor_paths}) == n_saved_cases_expected))

display(Markdown("### Checks"))
for label, ok in checks:
    print(f"{'OK' if ok else 'FAIL':4s}  {label}")

display(Markdown("### Metadata"))
display(metadata)

display(Markdown("### run.log tail"))
print(log_text)

## 2. Tensor Archive Audit

This validates a few saved `.npz` archives. It checks shapes, finite values, and the conditioning identity `observed ~= truth * mask` for zero-noise runs.

In [ ]:
print("n tensor archives =", len(tensor_paths))
for path in tensor_paths[:12]:
    print(path.relative_to(OUT_DIR))
if len(tensor_paths) > 12:
    print(f"... {len(tensor_paths) - 12} more")

audit_paths = tensor_paths[:2] + tensor_paths[len(tensor_paths)//2:len(tensor_paths)//2+1] + tensor_paths[-2:]
audit_rows = []
for path in audit_paths:
    z = np.load(path, allow_pickle=False)
    ensemble = z["ensemble"]
    truth = z["truth"]
    mask = z["mask"]
    observed = z["observed"]
    obs_expected = truth * mask[None, :, :]
    audit_rows.append({
        "file": str(path.relative_to(OUT_DIR)),
        "ensemble_shape": tuple(ensemble.shape),
        "truth_shape": tuple(truth.shape),
        "mask_shape": tuple(mask.shape),
        "observed_shape": tuple(observed.shape),
        "all_finite": bool(np.isfinite(ensemble).all() and np.isfinite(truth).all() and np.isfinite(mask).all()),
        "mask_fraction": float(mask.mean()),
        "observed_truth_mask_max_abs": float(np.max(np.abs(observed - obs_expected))),
        "ensemble_conc_min": float(np.min(ensemble[:, 0])),
        "ensemble_conc_max": float(np.max(ensemble[:, 0])),
        "ensemble_thick_min": float(np.min(ensemble[:, 1])) if ensemble.shape[1] > 1 else np.nan,
        "ensemble_thick_max": float(np.max(ensemble[:, 1])) if ensemble.shape[1] > 1 else np.nan,
    })

if pd is not None:
    display(pd.DataFrame(audit_rows))
else:
    display(audit_rows)

## 3. Aggregate Scores

Metrics are grouped by variable and mask type. For the recommended run, `eval_region` should be `unobserved`, so the diagnostics evaluate reconstruction away from the observed tracks.

In [ ]:
agg = read_csv_table(OUT_DIR / "aggregate_metrics.csv")
display_table(agg, n=30)

agg_rows = table_rows(agg)
score_cols = ["rmse", "crps", "coverage_0.5", "coverage_0.8", "coverage_0.9", "coverage_0.95", "spread", "skill_rmse", "spread_skill_ratio"]
if agg_rows:
    for row in agg_rows:
        variable = row.get("variable")
        mask_type = row.get("mask_type")
        eval_region = row.get("eval_region", "unknown")
        print(f"\n{variable} / {mask_type} / eval={eval_region}")
        for col in score_cols:
            if col in row:
                print(f"  {col:20s} {to_float(row[col]):.6g}")

## 4. Coverage Reliability

In [ ]:
coverage_cols = ["coverage_0.5", "coverage_0.8", "coverage_0.9", "coverage_0.95"]
rows = [r for r in agg_rows if all(c in r for c in coverage_cols)]
if rows:
    fig, ax = plt.subplots(figsize=(7, 6))
    nominal = np.array([0.5, 0.8, 0.9, 0.95])
    for row in rows:
        empirical = np.array([to_float(row[c]) for c in coverage_cols])
        label = f"{row.get('variable')} / {row.get('mask_type')}"
        ax.plot(nominal, empirical, marker="o", linewidth=2, label=label)
        for x, y in zip(nominal, empirical):
            ax.text(x, y, f"{y:.2f}", fontsize=8)
    ax.plot([0.45, 1.0], [0.45, 1.0], "k--", linewidth=1, label="ideal")
    ax.set_xlabel("nominal central interval")
    ax.set_ylabel("empirical coverage")
    ax.set_title("Coverage reliability")
    ax.set_xlim(0.45, 1.0)
    ax.set_ylim(0.0, 1.02)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()
else:
    print("coverage columns are missing")

## 5. Rank Histograms

A calibrated marginal ensemble should be close to flat. U-shape means underdispersion; center-heavy means overdispersion; left/right tilt indicates bias. The chi-square-like value below is only a shape score, not a valid p-value, because pixels are spatially dependent.

In [ ]:
rank_path = ARRAYS_DIR / "rank_histograms.npz"
rank_rows = []
if rank_path.exists():
    rank_data = np.load(rank_path)
    keys = sorted(rank_data.files)
    fig, axes = plt.subplots(len(keys), 1, figsize=(11, max(3.3, 3.1 * len(keys))), squeeze=False)
    for ax, key in zip(axes.ravel(), keys):
        counts = rank_data[key].astype(np.float64)
        total = counts.sum()
        probs = counts / total if total else counts
        expected = 1.0 / len(counts)
        ranks = np.arange(len(counts))
        ax.bar(ranks, probs, color="black")
        ax.axhline(expected, color="tab:red", linestyle="--", linewidth=1.2, label="flat")
        ax.set_title(key)
        ax.set_xlabel("rank of truth among ensemble members")
        ax.set_ylabel("probability")
        ax.grid(True, axis="y", alpha=0.25)
        ax.legend()

        edge_ratio = float((probs[0] + probs[-1]) / (2 * expected)) if expected else np.nan
        center_slice = probs[len(probs)//3 : 2*len(probs)//3]
        center_ratio = float(center_slice.mean() / expected) if expected else np.nan
        bias_index = float((probs[-1] - probs[0]) / expected) if expected else np.nan
        chi2_like = float(((counts - total * expected) ** 2 / max(total * expected, 1.0)).sum()) if total else np.nan
        rank_rows.append({
            "key": key,
            "bins": len(counts),
            "total_ranks": int(total),
            "flat_probability": expected,
            "edge_ratio_vs_flat": edge_ratio,
            "center_ratio_vs_flat": center_ratio,
            "right_minus_left_edges_in_flat_units": bias_index,
            "chi2_like_not_independent": chi2_like,
        })
    plt.tight_layout()
    plt.show()
    if pd is not None:
        display(pd.DataFrame(rank_rows))
    else:
        display(rank_rows)
else:
    print("missing:", rank_path)

## 6. Spread-Skill And Per-Case Distributions

In [ ]:
per_case = read_csv_table(OUT_DIR / "per_case_metrics.csv")
per_rows = table_rows(per_case)
display_table(per_case, n=8)

if per_rows:
    variables = sorted({r.get("variable", "unknown") for r in per_rows})
    metrics = ["rmse", "crps", "spread_skill_ratio", "observed_fraction", "evaluated_fraction"]
    for metric in metrics:
        if metric not in per_rows[0]:
            continue
        fig, ax = plt.subplots(figsize=(8, 4))
        for variable in variables:
            vals = [to_float(r[metric]) for r in per_rows if r.get("variable") == variable]
            vals = np.array([v for v in vals if np.isfinite(v)])
            if vals.size:
                ax.hist(vals, bins=24, alpha=0.55, label=variable)
        ax.set_title(metric)
        ax.set_xlabel(metric)
        ax.set_ylabel("condition-case rows")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()
else:
    print("per-case metrics are missing")

## 7. Saved Tensor Preview

Set `ARCHIVE_INDEX` to inspect a different case/conditioning archive.

In [ ]:
ARCHIVE_INDEX = 0
CHANNEL = 0

if tensor_paths:
    path = tensor_paths[min(ARCHIVE_INDEX, len(tensor_paths) - 1)]
    z = np.load(path, allow_pickle=False)
    ensemble = z["ensemble"]
    truth = z["truth"]
    mask = z["mask"]
    observed = z["observed"]
    mean = ensemble.mean(axis=0)
    spread = ensemble.std(axis=0)

    print("archive:", path.relative_to(OUT_DIR))
    for key in z.files:
        arr = z[key]
        print(f"{key:16s} shape={arr.shape} dtype={arr.dtype}")

    fig, axes = plt.subplots(2, 4, figsize=(15, 7))
    panels = [
        (truth[CHANNEL], "truth", "viridis"),
        (observed[CHANNEL], "observed on tracks", "viridis"),
        (mask, "track mask", "gray"),
        (mean[CHANNEL], "ensemble mean", "viridis"),
        (ensemble[0, CHANNEL], "sample 0", "viridis"),
        (ensemble[min(1, ensemble.shape[0]-1), CHANNEL], "sample 1", "viridis"),
        (spread[CHANNEL], "ensemble spread", "magma"),
        (mean[CHANNEL] - truth[CHANNEL], "mean - truth", "coolwarm"),
    ]
    for ax, (img, title, cmap) in zip(axes.ravel(), panels):
        im = ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()
else:
    print("No saved tensors found. Rerun with --save-tensors.")

## 8. Saved CLI Plots

In [ ]:
display(Markdown("### Example panels"))
show_many("example_*.png", width=1050, limit=8)

display(Markdown("### Rank histogram plots"))
show_many("rank_hist_*.png", width=950)

display(Markdown("### Coverage plots"))
show_many("coverage_*.png", width=950)

display(Markdown("### Spread-skill plots"))
show_many("spread_skill_*.png", width=950)

display(Markdown("### Metric-vs-density plots"))
show_many("rmse_vs_density.png", width=950)
show_many("crps_vs_density.png", width=950)

## 9. Trust And Limitations

Use this run as a targeted diagnostic of swath-conditioned reconstruction, not as a final proof of posterior calibration.

What is reasonably trustworthy:

- the tensors are saved and auditable;
- the condition is track-based swath conditioning;
- metrics/rank/coverage are computed on `eval_region=unobserved` if the run used the recommended command;
- strong failures such as U-shaped rank histograms, bad coverage, or spread-skill ratio far from 1 are meaningful signals.

Main limitations:

- 30 days is a short sample and may not cover seasonal regimes;
- rank histogram bins are built from spatial pixels, which are correlated, so the chi-square-like score is not a formal p-value;
- rank histograms check marginal calibration, not full joint spatial realism;
- multiple swath repeats per day help coverage of conditions but are not independent days.

For a stronger claim, repeat the same setup with more days, keep `--eval-region unobserved`, and compare against another method/baseline on exactly the same saved masks.